# AI Commerce Intelligence Platform
## Notebook 03: Dynamic ABC/XYZ Portfolio Segmentation

**Objective**: Classify the entire product catalog across two primary axes:
1. **ABC Analysis**: Commercial value contribution (Pareto 80/20 revenue rule).
2. **XYZ Analysis**: Demand predictability and volatility (Coefficient of Variation on unconstrained demand).
3. **9-Box Operational Matrix**: Combined segmentation defining tailored replenishment, forecasting, and inventory policies.


In [ ]:
import os
import sys
from pathlib import Path

workspace_dir = Path.cwd().resolve()
if workspace_dir.name == "notebooks":
    workspace_dir = workspace_dir.parent
sys.path.insert(0, str(workspace_dir / "src"))

mpl_cache = workspace_dir / ".gemini" / "scratch" / "mpl"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(mpl_cache)

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from commerce_ai.data.loaders import CommerceDataLoader
from commerce_ai.analytics.abc_xyz import (
    ABCConfig,
    XYZConfig,
    calculate_abc_classification,
    calculate_xyz_classification,
    calculate_abc_xyz_matrix,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 10


### 1. Ingest Catalog & Transactional Sales

In [ ]:
data_dir = workspace_dir / "data" / "sample"
loader = CommerceDataLoader()
datasets = loader.load_all(data_dir, validate=False, validate_relational=False)

sales_df = datasets["sales"]
products_df = datasets["products"]
print(f"Catalog contains {len(products_df):,d} SKUs with ${sales_df['revenue'].sum():,.2f} in historical revenue.")


### 2. Revenue Pareto Distribution (ABC Analysis)
Auditing how commercial revenue is concentrated across the SKU catalog.

In [ ]:
abc_cfg = ABCConfig(a_threshold=0.80, b_threshold=0.95, metric="revenue")
abc_results = calculate_abc_classification(sales_df, config=abc_cfg)

fig, ax1 = plt.subplots(figsize=(12, 5))
x = np.arange(len(abc_results))

# Revenue bar plot
ax1.bar(x, abc_results["revenue"], color="steelblue", alpha=0.6, label="SKU Revenue")
ax1.set_ylabel("Total Revenue ($)", color="steelblue")
ax1.set_xlabel("SKUs (Ranked by Revenue)")

# Cumulative share line
ax2 = ax1.twinx()
ax2.plot(x, abc_results["cumulative_revenue_share"] * 100, color="crimson", linewidth=2.5, label="Cumulative Share %")
ax2.axhline(80, color="orange", linestyle="--", alpha=0.8, label="Class A Cutoff (80%)")
ax2.axhline(95, color="green", linestyle="--", alpha=0.8, label="Class B Cutoff (95%)")
ax2.set_ylabel("Cumulative Revenue Share (%)", color="crimson")
ax2.set_ylim(0, 105)

plt.title("Cumulative Revenue Pareto Curve & ABC Thresholds")
ax1.legend(loc="center left")
ax2.legend(loc="center right")
plt.tight_layout()
plt.show()

# Print breakdown
abc_counts = abc_results["abc_class"].value_counts()
print("ABC SKU Distribution:")
for c in ["A", "B", "C"]:
    cnt = abc_counts.get(c, 0)
    rev = abc_results[abc_results["abc_class"] == c]["revenue"].sum()
    pct = rev / abc_results["revenue"].sum() * 100
    print(f"  Class {c}: {cnt:3d} SKUs ({cnt/len(abc_results)*100:5.1f}%) | Revenue: ${rev:12,.2f} ({pct:5.1f}%)")


### 3. Demand Volatility & Predictability (XYZ Analysis)
Classifying SKUs by Coefficient of Variation:
- **Class X** ($CV \le 0.50$): High predictability, steady demand.
- **Class Y** ($0.50 < CV \le 1.00$): Moderate variability, cyclical/promotional.
- **Class Z** ($CV > 1.00$): High volatility, sporadic or intermittent demand.


In [ ]:
xyz_cfg = XYZConfig(x_cv_threshold=0.50, y_cv_threshold=1.00, filter_stockouts=False)
xyz_results = calculate_xyz_classification(sales_df, config=xyz_cfg)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(xyz_results["coefficient_of_variation"].dropna(), bins=30, kde=True, ax=axes[0], color="darkcyan")
axes[0].axvline(0.5, color="orange", linestyle="--", label="Class X Cutoff (0.50)")
axes[0].axvline(1.0, color="crimson", linestyle="--", label="Class Y Cutoff (1.00)")
axes[0].set_title("Demand Coefficient of Variation (CV) Distribution")
axes[0].set_xlabel("CV (Std / Mean)")
axes[0].legend()

xyz_counts = xyz_results["xyz_class"].value_counts()
sns.barplot(x=xyz_counts.index, y=xyz_counts.values, hue=xyz_counts.index, order=["X", "Y", "Z"], ax=axes[1], palette="Blues_d", legend=False)
axes[1].set_title("XYZ SKU Count Breakdown")
axes[1].set_xlabel("Volatility Class")
axes[1].set_ylabel("SKU Count")
plt.tight_layout()
plt.show()

print("XYZ Breakdown:")
for c in ["X", "Y", "Z"]:
    cnt = xyz_counts.get(c, 0)
    print(f"  Class {c}: {cnt:3d} SKUs ({cnt/len(xyz_results)*100:5.1f}%)")


### 4. The 9-Box ABC-XYZ Matrix
Cross-tabulating commercial revenue importance with demand volatility.

In [ ]:
sku_matrix, summary_table, crosstab = calculate_abc_xyz_matrix(sales_df, abc_cfg, xyz_cfg)

# Heatmap of SKU counts in the 9-box
matrix_3x3 = crosstab.loc[["A", "B", "C"], ["X", "Y", "Z"]]

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(matrix_3x3, annot=True, fmt="d", cmap="YlGnBu", cbar=True, ax=ax, annot_kws={"size": 14, "weight": "bold"})
ax.set_title("ABC-XYZ 9-Box Matrix (SKU Count)")
ax.set_xlabel("Demand Volatility (XYZ)")
ax.set_ylabel("Revenue Value (ABC)")
plt.tight_layout()
plt.show()

print("Operational Summary Table:")
summary_table.style.format({
    "total_revenue": "${:,.2f}",
    "revenue_share": "{:.2%}",
    "avg_daily_demand": "{:.2f}",
    "avg_cv": "{:.3f}"
})


### 5. Strategic Business Interpretation

| Segment | Operational Profile | Recommended Forecasting & Replenishment Strategy |
| :---: | :--- | :--- |
| **AX** | **High Value + High Predictability** | Core revenue drivers. Highly suited for automated statistical / ML forecasting. Low safety stock buffer needed. Automated PO issuance. |
| **AY** | **High Value + Moderate Variability** | Seasonal or promotional revenue drivers. LightGBM / TimesFM-3 with calendar promotional features. Moderate safety stock buffers. |
| **AZ** | **High Value + Erratic / Volatile** | High commercial impact with unpredictable demand. Exception-based forecasting with human planner review. Tighter supplier SLA monitoring. |
| **BX** | **Medium Value + High Predictability** | Steady catalog staples. Standard automated replenishment with fixed periodic review. |
| **BY** | **Medium Value + Moderate Variability** | Mid-tier seasonal goods. Dynamic safety stock based on lead time variability. |
| **BZ** | **Medium Value + Erratic** | Lumpy demand. Reorder-point policies with Croston / zero-inflated demand models. |
| **CX** | **Low Value + High Predictability** | Low risk, steady turnover. Bulk ordering to satisfy supplier MOQs and maximize volume freight discounts. |
| **CY** | **Low Value + Moderate Variability** | Long-tail standard items. Low safety stock priority; periodic review. |
| **CZ** | **Low Value + Erratic Demand** | Long-tail / obsolete candidates. Danger zone for dead stock. Consider made-to-order, drop-shipping, or catalog rationalization. |
